# Two Populations — Disentangled Risk Factors

Two populations share the same **N** Gaussian features but are driven by different latent risk directions separated by angle **θ**:

- **Population A**: risk = `w₁ · x`
- **Population B**: risk = `w₂ · x`,  where `w₁ · w₂ = cos(θ)`

We compare five models on three evaluation slices (pop A, pop B, combined).

## 0. Parameters

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys, pathlib, importlib

import claimssimulator as csim
from claimssimulator.metrics import gini, calibration_quality_ratio as cqr, r2, rmse, mape

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# ── Experiment parameters ─────────────────────────────────────
N_FEATURES       = 5         # number of Gaussian features
N_SAMPLES_A      = 5_000      # policies in population A
N_SAMPLES_B      = 10_000      # policies in population B
CONTRACT_MEAN    = 3.0        # mean contract duration (years, Exponential)
CONTRACT_MAX     = 10.0       # cap on contract duration
THETA_DEG        = 30         # angle (degrees) between the two risk directions
FREQ_A           = 0.10       # base claim frequency for population A
FREQ_B           = 0.07       # base claim frequency for population B
NOISE_RATIO      = 0.4       # fraction of log-risk variance from unobserved noise
                              # (0.0 = fully observable, 0.5 = half hidden)
SEED             = 42

ModuleNotFoundError: No module named 'claimsimulator'

## 1. Risk Directions

Build two unit vectors **w₁** and **w₂** in ℝᴺ separated by angle θ.

In [ ]:
theta = np.deg2rad(THETA_DEG)
rng = np.random.default_rng(SEED)

w1 = rng.standard_normal(N_FEATURES)
w1 = w1 / np.linalg.norm(w1)

w_rand = rng.standard_normal(N_FEATURES)
w_perp = w_rand - w_rand.dot(w1) * w1
w_perp = w_perp / np.linalg.norm(w_perp)
w2 = np.cos(theta) * w1 + np.sin(theta) * w_perp

print(f"w1 · w2 = {w1.dot(w2):.4f}  (expected cos({THETA_DEG}°) = {np.cos(theta):.4f})")

NameError: name 'np' is not defined

## 2. Generate Features & Simulate Claims

Both populations share the same feature distributions. Each gets its own `ClaimsSimulator` driven by its respective risk factor.

In [ ]:
import claimssimulator.simulation
importlib.reload(claimssimulator.simulation)
from claimssimulator.simulation import ClaimSource, MultiCauseClaimsSimulator

feature_cols = [f'x{i}' for i in range(N_FEATURES)]

def make_features(n_samples, seed):
    spec = [csim.Feature(f'x{i}', csim.Normal(loc=0, scale=1)) for i in range(N_FEATURES)]
    spec.append(csim.Feature(
        'contract_duration_years',
        csim.Exponential(scale=CONTRACT_MEAN),
        csim.Transform(lambda x: np.clip(x, 0.1, CONTRACT_MAX)),
    ))
    return csim.FeatureDefinition(spec).generate(n_samples=n_samples, random_seed=seed)

def simulate_claims(feat_df, w, freq, seed):
    """Single-source Poisson simulation via MultiCauseClaimsSimulator."""
    local_rng = np.random.default_rng(seed)
    X = feat_df[feature_cols].values
    feat_df = feat_df.copy()
    signal = X @ w
    # Add unobserved heterogeneity controlled by NOISE_RATIO
    noise = local_rng.standard_normal(len(feat_df))
    feat_df['risk_score'] = np.sqrt(1 - NOISE_RATIO) * signal + np.sqrt(NOISE_RATIO) * noise
    feat_df['rate']       = freq * np.exp(feat_df['risk_score'].values)

    source = ClaimSource(name='claim', generator='Poisson',
                         param_columns={'rate': 'rate'})
    sim = MultiCauseClaimsSimulator(
        sources=[source],
        time_to_simulate='contract_duration_years',
        max_exposure=1.0,
        exposure_column='exposure',
        claim_counter='past_claims',
        renewal_mode='contract_end',
        random_seed=seed,
    )
    out = sim.simulate(feat_df)
    out = out.rename(columns={'claim_claim': 'claim'})
    return out

# ── Population A (risk driven by w1) ────────────────────────
feat_a = make_features(N_SAMPLES_A, seed=SEED)
df_a   = simulate_claims(feat_a, w1, FREQ_A, seed=SEED)
df_a['population'] = 'A'

# ── Population B (risk driven by w2) ────────────────────────
feat_b = make_features(N_SAMPLES_B, seed=SEED + 1)
df_b   = simulate_claims(feat_b, w2, FREQ_B, seed=SEED + 1)
df_b['population'] = 'B'

# ── Combined ─────────────────────────────────────────────────
df_all = pd.concat([df_a, df_b], ignore_index=True)

print(f"NOISE_RATIO = {NOISE_RATIO}  →  {(1-NOISE_RATIO)*100:.0f}% of log-risk variance is observable")
for label, d in [('A', df_a), ('B', df_b), ('all', df_all)]:
    print(f"Pop {label:3s}  rows={len(d):,}  claims={d['claim'].sum():,}  "
          f"rate={d['claim'].sum()/d['exposure'].sum():.4f}")

NOISE_RATIO = 0.5  →  50% of log-risk variance is observable
Pop A    rows=19,525  claims=2,560  rate=0.1774
Pop B    rows=19,621  claims=2,564  rate=0.1766
Pop all  rows=39,146  claims=5,124  rate=0.1770


## 3. Train / Test Split

We split each population separately (stratified by population), then rebuild the combined sets.

In [ ]:
from sklearn.model_selection import train_test_split

def split_by_contract(df, test_size=0.25, seed=SEED):
    contracts = df['contract_id'].unique()
    tr, te = train_test_split(contracts, test_size=test_size, random_state=seed)
    return df[df['contract_id'].isin(tr)].copy(), df[df['contract_id'].isin(te)].copy()

train_a, test_a = split_by_contract(df_a)
train_b, test_b = split_by_contract(df_b)

train_all = pd.concat([train_a, train_b], ignore_index=True)
test_all  = pd.concat([test_a,  test_b],  ignore_index=True)

for label, tr, te in [('A', train_a, test_a), ('B', train_b, test_b), ('all', train_all, test_all)]:
    print(f"Pop {label:3s}  train={len(tr):,} ({tr['claim'].sum():,} claims)  "
          f"test={len(te):,} ({te['claim'].sum():,} claims)")

Pop A    train=14,620 (1,905 claims)  test=4,905 (655 claims)
Pop B    train=14,666 (1,867 claims)  test=4,955 (697 claims)
Pop all  train=29,286 (3,772 claims)  test=9,860 (1,352 claims)


## 4. Train Five Models

| Model | Training data | Description |
|---|---|---|
| `ebm_all` | combined | Single model, sees both populations |
| `ebm_a` | pop A only | Specialist for population A |
| `ebm_b` | pop B only | Specialist for population B |
| `ebm_res_a` | pop A only | Residual on `ebm_all`, specialises on A |
| `ebm_res_b` | pop B only | Residual on `ebm_all`, specialises on B |

The residual models use `log(ebm_all predictions)` as `init_score`, so they learn only the deviation from the global model on their population.

In [ ]:
EBM_PARAMS = dict(max_bins=64, max_rounds=300, learning_rate=0.05,
                  objective='poisson_deviance', random_state=SEED)

def fit_ebm(X, y, w):
    m = ExplainableBoostingRegressor(**EBM_PARAMS)
    m.fit(X, y, sample_weight=w)
    return m

def Xy(df):
    return df[feature_cols], df['claim'] / df['exposure'], df['exposure']

# ── 1. Global model ──────────────────────────────────────────
X_tr_all, y_tr_all, w_tr_all = Xy(train_all)
ebm_all = fit_ebm(X_tr_all, y_tr_all, w_tr_all)
print("ebm_all    ✓")

# ── 2. Population specialists ────────────────────────────────
X_tr_a, y_tr_a, w_tr_a = Xy(train_a)
X_tr_b, y_tr_b, w_tr_b = Xy(train_b)

ebm_a = fit_ebm(X_tr_a, y_tr_a, w_tr_a)
print("ebm_a      ✓")

ebm_b = fit_ebm(X_tr_b, y_tr_b, w_tr_b)
print("ebm_b      ✓")

# ── 3. Residual models ───────────────────────────────────────
# With a log-link Poisson model, using ebm_all as offset is equivalent to
# fitting on the rate-ratio:  y_ratio = y / ebm_all_pred  (mean ≈ 1).
# Final prediction: ebm_all.predict(X) * ebm_res.predict(X)

y_ratio_a = y_tr_a / np.clip(ebm_all.predict(X_tr_a), 1e-9, None)
ebm_res_a = fit_ebm(X_tr_a, y_ratio_a, w_tr_a)
print("ebm_res_a  ✓  (ratio mean: "
      f"{np.average(y_ratio_a, weights=w_tr_a):.4f})")

y_ratio_b = y_tr_b / np.clip(ebm_all.predict(X_tr_b), 1e-9, None)
ebm_res_b = fit_ebm(X_tr_b, y_ratio_b, w_tr_b)
print("ebm_res_b  ✓  (ratio mean: "
      f"{np.average(y_ratio_b, weights=w_tr_b):.4f})")

# ── 4. Global model with population as categorical feature ────
# Add a 'population' string column so EBM treats it as categorical.
feature_cols_pop = feature_cols + ['population']

def make_pop_X(df, pop_label):
    X = df[feature_cols].copy()
    X['population'] = pop_label
    return X

X_tr_all_pop = pd.concat([
    make_pop_X(train_a, 'A'),
    make_pop_X(train_b, 'B'),
], ignore_index=True)
y_tr_all_pop = pd.concat([y_tr_a, y_tr_b], ignore_index=True)
w_tr_all_pop = pd.concat([w_tr_a, w_tr_b], ignore_index=True)

ebm_all_pop = fit_ebm(X_tr_all_pop, y_tr_all_pop, w_tr_all_pop)
print("ebm_all_pop ✓")


NameError: name 'ExplainableBoostingRegressor' is not defined

## 5. Performance Tables

**Rows** = evaluation slice: pop A test, pop B test, combined test  
**Columns** = model predictions  

For the residual models the final prediction is:
`exp(init_score + EBM_residual_output)` = `ebm_all_pred × exp(residual)`

In [ ]:
def predict_residual(ebm_base, ebm_res, X):
    """Final rate = base_pred × residual_multiplier."""
    return ebm_base.predict(X) * np.clip(ebm_res.predict(X), 1e-9, None)

def make_tables(test_slices, model_preds):
    gini_rows, cqr_rows = {}, {}
    for slice_name, (y_true, w) in test_slices.items():
        grow, crow = {}, {}
        for model_name, preds_by_slice in model_preds.items():
            p = preds_by_slice[slice_name]
            grow[model_name] = gini(y_true, p, w)
            crow[model_name] = cqr(y_true, p, w, n_bins=20, method='mse', norm='sqrt')
        gini_rows[slice_name] = grow
        cqr_rows[slice_name]  = crow
    return pd.DataFrame(gini_rows).T, pd.DataFrame(cqr_rows).T

def make_rate_tables(rate_slices, model_preds):
    r2_rows, rmse_rows, mape_rows = {}, {}, {}
    for slice_name, (y_rate, w) in rate_slices.items():
        rrow, mrow, prow = {}, {}, {}
        for model_name, preds_by_slice in model_preds.items():
            p = preds_by_slice[slice_name]
            rrow[model_name] = r2(y_rate, p, w)
            mrow[model_name] = rmse(y_rate, p, w)
            prow[model_name] = mape(y_rate, p, w)
        r2_rows[slice_name]   = rrow
        rmse_rows[slice_name] = mrow
        mape_rows[slice_name] = prow
    return pd.DataFrame(r2_rows).T, pd.DataFrame(rmse_rows).T, pd.DataFrame(mape_rows).T

# ── Test-set feature matrices ────────────────────────────────
X_te_a   = test_a[feature_cols]
X_te_b   = test_b[feature_cols]
X_te_all = test_all[feature_cols]

# Feature matrices with population column
X_te_a_pop   = make_pop_X(test_a, 'A')
X_te_b_pop   = make_pop_X(test_b, 'B')
X_te_all_pop = pd.concat([make_pop_X(test_a, 'A'), make_pop_X(test_b, 'B')], ignore_index=True)

# ── Evaluation slices: (claims, exposure) ────────────────────
test_slices = {
    'pop_A':    (test_a['claim'].values,   test_a['exposure'].values),
    'pop_B':    (test_b['claim'].values,   test_b['exposure'].values),
    'combined': (test_all['claim'].values, test_all['exposure'].values),
}

# ── True rate slices (noise-inclusive underlying rate) ───────
rate_slices = {
    'pop_A':    (test_a['rate'].values,   test_a['exposure'].values),
    'pop_B':    (test_b['rate'].values,   test_b['exposure'].values),
    'combined': (test_all['rate'].values, test_all['exposure'].values),
}

# ── Predictions per model per slice ─────────────────────────
model_preds = {
    'ebm_all': {
        'pop_A':    ebm_all.predict(X_te_a),
        'pop_B':    ebm_all.predict(X_te_b),
        'combined': ebm_all.predict(X_te_all),
    },
    'ebm_all_pop': {
        'pop_A':    ebm_all_pop.predict(X_te_a_pop),
        'pop_B':    ebm_all_pop.predict(X_te_b_pop),
        'combined': ebm_all_pop.predict(X_te_all_pop),
    },
    'ebm_a': {
        'pop_A':    ebm_a.predict(X_te_a),
        'pop_B':    ebm_a.predict(X_te_b),
        'combined': ebm_a.predict(X_te_all),
    },
    'ebm_b': {
        'pop_A':    ebm_b.predict(X_te_a),
        'pop_B':    ebm_b.predict(X_te_b),
        'combined': ebm_b.predict(X_te_all),
    },
    'ebm_res_a': {
        'pop_A':    predict_residual(ebm_all, ebm_res_a, X_te_a),
        'pop_B':    predict_residual(ebm_all, ebm_res_a, X_te_b),
        'combined': predict_residual(ebm_all, ebm_res_a, X_te_all),
    },
    'ebm_res_b': {
        'pop_A':    predict_residual(ebm_all, ebm_res_b, X_te_a),
        'pop_B':    predict_residual(ebm_all, ebm_res_b, X_te_b),
        'combined': predict_residual(ebm_all, ebm_res_b, X_te_all),
    },
}

gini_table, cqr_table = make_tables(test_slices, model_preds)
r2_table, rmse_table, mape_table = make_rate_tables(rate_slices, model_preds)

print("=== Gini (vs observed claims) ===")
display(gini_table.style.format("{:.4f}").background_gradient(cmap='RdYlGn', axis=1))

print("\n=== CQR (n_bins=20, mse, sqrt) — vs observed claims ===")
display(cqr_table.style.format("{:.4f}").background_gradient(cmap='RdYlGn', axis=1))

print("\n=== R² (vs true underlying rate) ===")
display(r2_table.style.format("{:.4f}").background_gradient(cmap='RdYlGn', axis=1))

print("\n=== RMSE (vs true underlying rate) ===")
display(rmse_table.style.format("{:.4f}").background_gradient(cmap='RdYlGn_r', axis=1))

print("\n=== MAPE (vs true underlying rate) ===")
display(mape_table.style.format("{:.4f}").background_gradient(cmap='RdYlGn_r', axis=1))

=== Gini ===


,ebm_all,ebm_all_pop,ebm_a,ebm_b,ebm_res_a,ebm_res_b
pop_A,0.5338,0.5428,0.5432,0.4881,0.5408,0.5111
pop_B,0.5236,0.5431,0.4693,0.5430,0.4775,0.5407
combined,0.5288,0.5431,0.5075,0.5144,0.5099,0.5254



=== CQR (n_bins=20, mse, sqrt) ===


,ebm_all,ebm_all_pop,ebm_a,ebm_b,ebm_res_a,ebm_res_b
pop_A,0.2928,0.3045,0.3043,0.2780,0.3015,0.2846
pop_B,0.2587,0.2721,0.2459,0.2720,0.2447,0.2682
combined,0.2916,0.3075,0.2894,0.2936,0.2875,0.2943


In [ ]:
import pickle
from datetime import datetime
from pathlib import Path

RESULTS_PATH = Path('../../data/two_populations_experiments.pkl')

experiment = {
    'timestamp': datetime.now().isoformat(),
    'params': {
        'N_FEATURES':   N_FEATURES,
        'N_SAMPLES_A':  N_SAMPLES_A,
        'N_SAMPLES_B':  N_SAMPLES_B,
        'CONTRACT_MEAN': CONTRACT_MEAN,
        'CONTRACT_MAX':  CONTRACT_MAX,
        'THETA_DEG':    THETA_DEG,
        'FREQ_A':       FREQ_A,
        'FREQ_B':       FREQ_B,
        'SEED':         SEED,
    },
    'w1': w1,
    'w2': w2,
    'gini': gini_table,
    'cqr':  cqr_table,
}

if RESULTS_PATH.exists():
    with open(RESULTS_PATH, 'rb') as f:
        all_results = pickle.load(f)
else:
    all_results = []

all_results.append(experiment)

with open(RESULTS_PATH, 'wb') as f:
    pickle.dump(all_results, f)

print(f"Saved experiment #{len(all_results)}  →  {RESULTS_PATH}")
print(f"  params: {experiment['params']}")

Saved experiment #2  →  ../../data/two_populations_experiments.pkl
  params: {'N_FEATURES': 6, 'N_SAMPLES_A': 5000, 'N_SAMPLES_B': 5000, 'CONTRACT_MEAN': 3.0, 'CONTRACT_MAX': 10.0, 'THETA_DEG': 30, 'FREQ_A': 0.1, 'FREQ_B': 0.1, 'SEED': 42}
